# Notebook 04 — Thesis Figures
**Thesis:** Chapter 7 | **Input:** `data/2_events.csv`, `data/3_*.csv` | **Output:** `figures/fig_*.pdf + .png`

---

## Complete Figure Map

| Figure | Content | Event | Section |
|--------|---------|-------|---------|
| Fig 7.1 | PDR decomposition system→raw→link | — | §7.1.1 |
| Fig 7.2 | Loss zone breakdown | — | §7.1.1 |
| Fig 7.3 | Rolling PDR time series all 6 devices | — | §7.1.2 |
| Fig 7.4 | Burst loss distribution | E16 | §7.2.1 |
| Fig 7.5 | Weekday vs weekend PDR per device | E3 | §7.3.1 |
| Fig 7.6 | PDR by time of day | E4 | §7.3.2 |
| Fig 7.7 | PDR by season | E6 | §7.3.3 |
| Fig 7.8 | CO₂ vs rolling PDR dual axis (ED3) | E1/E2 | §7.4 |
| Fig 7.9 | CO₂ tier PDR + burst rate | E1 | §7.4 |
| Fig 7.10 | PM2.5 spike and tier PDR | E7, E8 | §7.5.1 |
| Fig 7.11 | Pressure tier PDR | E9 | §7.5.2 |
| Fig 7.12 | Humidity tier PDR | E11 | §7.5.3 |
| Fig 7.13 | Temperature tier PDR | E12 | §7.5.3 |
| Fig 7.14 | RSSI tier PDR | E17 | §7.5 |
| Fig 7.15 | ESP tier PDR | E18 | §7.5 |
| Fig 7.16 | SF PDR + burst rate | E13 | §7.6.1 |
| Fig 7.17 | SF tier PDR comparison | E14, E15 | §7.6.1 |
| Fig 7.18 | SF×CO₂ heatmap — headline finding | E14×E1 | §7.6.2 |
| Fig 7.19 | Logistic regression forest plot | All | §7.7 |
| Fig 7.20 | Burst risk OR comparison | All | §7.7 |

All figures saved as **PDF** (Overleaf) and **PNG** (preview).

## 0 · Imports, Style & Load

In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

DATA_DIR = Path('../../data')
FIG_DIR  = Path('../figures'); FIG_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'font.family':'serif','font.size':10,'axes.titlesize':11,
    'axes.labelsize':10,'xtick.labelsize':9,'ytick.labelsize':9,
    'legend.fontsize':9,'figure.dpi':150,'savefig.dpi':300,
    'savefig.bbox':'tight','axes.spines.top':False,'axes.spines.right':False,
    'axes.grid':True,'grid.alpha':0.3,'grid.linestyle':'--',
})

C = {'blue':'#0077BB','orange':'#EE7733','green':'#009988',
     'red':'#CC3311','purple':'#AA3377','navy':'#004488','grey':'#BBBBBB'}
DEVICE_COLORS = [C['blue'],C['orange'],C['green'],C['red'],C['purple'],C['navy']]
TOA_MS = {7:71.9, 8:133.6, 9:246.8, 10:452.6}

def save_fig(fig, name):
    fig.savefig(FIG_DIR/f'{name}.pdf'); fig.savefig(FIG_DIR/f'{name}.png')
    print(f'  Saved: {name}')
    plt.close(fig)

def pdr(grp):
    rx=len(grp); lost=int(grp['mac_to_radio_loss'].sum())
    return rx/(rx+lost)*100 if (rx+lost)>0 else 0

def burst_rate(grp):
    return (grp['mac_to_radio_loss']>=3).mean()*100

def simple_bar(ax, labels, values, colors, title, ylabel, ylim=None, mean_line=None):
    bars = ax.bar(labels, values, color=colors, width=0.45, alpha=0.88, edgecolor='white')
    for bar, v in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.02, f'{v:.2f}%',
                ha='center', va='bottom', fontsize=8)
    if ylim: ax.set_ylim(ylim)
    if mean_line: ax.axhline(y=mean_line, color='black', lw=0.8, ls=':',
                             alpha=0.6, label=f'Mean ({mean_line:.2f}%)')
    ax.set_ylabel(ylabel); ax.set_title(title)
    if mean_line: ax.legend()

df = pd.read_csv(DATA_DIR/'2_events.csv')
df['time'] = pd.to_datetime(df['time'], utc=True)
df = df.sort_values(['device_id','time']).reset_index(drop=True)
df_link = df[~df['is_outage']&~df['is_sf_artifact']].copy().reset_index(drop=True)
or_table = pd.read_csv(DATA_DIR/'3_odds_ratios.csv')
DEVICES = sorted(df['device_id'].unique())

rx=len(df); lost_system=int(df['total_loss'].sum())
lost_raw=int(df['mac_to_radio_loss'].sum())
lost_link=int(df_link['mac_to_radio_loss'].sum())
pdr_system=rx/(rx+lost_system)*100
pdr_raw=rx/(rx+lost_raw)*100
pdr_link=len(df_link)/(len(df_link)+lost_link)*100
print(f'df_link: {len(df_link):,} rows | PDR_link: {pdr_link:.2f}%')

df_link: 1,156,407 rows | PDR_link: 96.36%


## 1 · Fig 7.1 — PDR Decomposition  *(§7.1.1)*

In [26]:
fig, ax = plt.subplots(figsize=(7,4.5))
labels=['PDR_system\n(incl. app-MAC drops)','PDR_raw\n(radio, all losses)','PDR_link\n(radio, env. only)']
values=[pdr_system,pdr_raw,pdr_link]
colors=[C['red'],C['orange'],C['green']]
bars=ax.bar(labels,values,color=colors,width=0.45,alpha=0.88,edgecolor='white')
for bar,v in zip(bars,values):
    ax.text(bar.get_x()+bar.get_width()/2,v+0.5,f'{v:.2f}%',ha='center',va='bottom',fontsize=10,fontweight='bold')
for i in range(len(values)-1):
    ax.annotate('',xy=(i+1,values[i+1]+1),xytext=(i,values[i]+1),
                arrowprops=dict(arrowstyle='->',color='black',lw=1.2))
ax.set_ylabel('PDR (%)')
ax.set_title('Fig 7.1 — PDR Decomposition: System → Raw → Link')
ax.set_ylim(50,103)
fig.tight_layout(); save_fig(fig,'fig_7_01_pdr_decomposition')

  Saved: fig_7_01_pdr_decomposition


## 2 · Fig 7.2 — Loss Zone Breakdown  *(§7.1.1)*

In [27]:
zones=['no_loss','sf_artifact','radio_loss','ambiguous','outage']
zlabels=['No loss','SF artifact','Radio loss (1–9)','Ambiguous (10–59)','Outage (≥60)']
zcolors=[C['green'],C['orange'],C['blue'],C['grey'],C['red']]
counts=df['loss_zone'].value_counts().reindex(zones).fillna(0)
pcts=counts/len(df)*100
fig,ax=plt.subplots(figsize=(9,3.5))
left=0
for zone,label,color in zip(zones,zlabels,zcolors):
    w=pcts[zone]
    ax.barh(0,w,left=left,color=color,edgecolor='white',height=0.5,label=f'{label} ({w:.2f}%)')
    if w>1.5: ax.text(left+w/2,0,f'{w:.1f}%',ha='center',va='center',fontsize=8,fontweight='bold',color='white')
    left+=w
ax.set_xlim(0,100); ax.set_yticks([])
ax.set_xlabel('Percentage of all intervals (%)')
ax.set_title('Fig 7.2 — Loss Zone Breakdown (all 1,217,313 intervals)')
ax.legend(loc='upper center',bbox_to_anchor=(0.5,-0.22),ncol=5,fontsize=8)
fig.tight_layout(); save_fig(fig,'fig_7_02_loss_zone_breakdown')

  Saved: fig_7_02_loss_zone_breakdown


## 3 · Fig 7.3 — Rolling PDR Time Series  *(§7.1.2)*

In [28]:
fig,axes=plt.subplots(6,1,figsize=(12,14),sharex=True)
for i,dev in enumerate(DEVICES):
    g=df_link[df_link['device_id']==dev].copy().set_index('time').sort_index()
    g['rx1']=1
    rpdr=(g['rx1'].rolling('30min',min_periods=1).sum()/
          g['total_tx'].rolling('30min',min_periods=1).sum()*100).clip(0,100)
    axes[i].plot(rpdr.index,rpdr.values,linewidth=0.4,color=DEVICE_COLORS[i],alpha=0.85)
    axes[i].set_ylabel(f'{dev}\nPDR (%)',fontsize=8)
    axes[i].set_ylim(0,100)
    axes[i].axhline(y=pdr_link,color='black',lw=0.6,ls=':',alpha=0.5)
    axes[i].set_yticks([0,50,100])
axes[-1].set_xlabel('Date')
axes[-1].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
plt.suptitle(f'Fig 7.3 — Rolling 30-min PDR per Device (Sep 2024–May 2025)\nDotted = PDR_link mean ({pdr_link:.2f}%)',fontsize=10,y=1.01)
fig.tight_layout(); save_fig(fig,'fig_7_03_rolling_pdr')

  Saved: fig_7_03_rolling_pdr


## 4 · Fig 7.4 — Burst Loss Distribution (E16)  *(§7.2.1)*

In [29]:
order=['no_loss','isolated','small_burst','large_burst']
olabels=['No loss','Isolated\n(1 packet)','Small burst\n(2–4 packets)','Large burst\n(≥5 packets)']
counts16=df_link['e16_loss_type'].value_counts().reindex(order).fillna(0)
pcts16=counts16/len(df_link)*100
fig,ax=plt.subplots(figsize=(7,4.5))
colors16=[C['green'],C['blue'],C['orange'],C['red']]
bars=ax.bar(olabels,pcts16.values,color=colors16,width=0.5,alpha=0.88,edgecolor='white')
for bar,v in zip(bars,pcts16.values):
    ax.text(bar.get_x()+bar.get_width()/2,v+0.02,f'{v:.2f}%',ha='center',va='bottom',fontsize=9)
ax.set_ylabel('Percentage of intervals (%)')
ax.set_title('Fig 7.4 — Burst Loss Distribution (E16)')
fig.tight_layout(); save_fig(fig,'fig_7_04_burst_distribution')

  Saved: fig_7_04_burst_distribution


## 5 · Fig 7.5 — Weekday vs Weekend PDR per Device (E3)  *(§7.3.1)*

In [30]:
wk_pdrs=[]; we_pdrs=[]
for dev in DEVICES:
    g=df_link[df_link['device_id']==dev]
    wk_pdrs.append(pdr(g[g['e3_is_weekday']==1]))
    we_pdrs.append(pdr(g[g['e3_is_weekday']==0]))
x=np.arange(len(DEVICES)); w=0.35
fig,ax=plt.subplots(figsize=(8,4.5))
b1=ax.bar(x-w/2,wk_pdrs,w,color=C['blue'],alpha=0.88,label='Weekday',edgecolor='white')
b2=ax.bar(x+w/2,we_pdrs,w,color=C['orange'],alpha=0.88,label='Weekend',edgecolor='white')
for bar,v in zip(b1,wk_pdrs): ax.text(bar.get_x()+bar.get_width()/2,v+0.02,f'{v:.1f}%',ha='center',va='bottom',fontsize=7.5)
for bar,v in zip(b2,we_pdrs): ax.text(bar.get_x()+bar.get_width()/2,v+0.02,f'{v:.1f}%',ha='center',va='bottom',fontsize=7.5)
ax.set_xticks(x); ax.set_xticklabels(DEVICES)
ax.set_ylabel('PDR_link (%)'); ax.set_title('Fig 7.5 — Weekday vs Weekend PDR per Device (E3)')
ax.set_ylim(94,98.5); ax.legend()
fig.tight_layout(); save_fig(fig,'fig_7_05_weekday_weekend_pdr')

  Saved: fig_7_05_weekday_weekend_pdr


## 6 · Fig 7.6 — PDR by Time of Day (E4)  *(§7.3.2)*

In [31]:
bands=['night','morning','peak','evening']
xlabs=['Night\n(00–06)','Morning\n(07–09)','Peak\n(10–17)','Evening\n(18–23)']
colors_tod=[C['navy'],C['blue'],C['red'],C['orange']]
pdrs_tod=[pdr(df_link[df_link['e4_time_of_day']==b]) for b in bands]
fig,ax=plt.subplots(figsize=(6,4.5))
simple_bar(ax,xlabs,pdrs_tod,colors_tod,'Fig 7.6 — PDR by Time of Day (E4)','PDR_link (%)',(94,98.5),pdr_link)
fig.tight_layout(); save_fig(fig,'fig_7_06_time_of_day_pdr')

  Saved: fig_7_06_time_of_day_pdr


## 7 · Fig 7.7 — PDR by Season (E6)  *(§7.3.3)*

In [32]:
seasons=['autumn','winter','spring']
slabels=['Autumn\n(Sep–Nov 2024)','Winter\n(Dec 2024–Feb 2025)','Spring\n(Mar–May 2025)']
scolors=[C['orange'],C['blue'],C['green']]
pdrs_s=[pdr(df_link[df_link['e6_season']==s]) for s in seasons]
fig,ax=plt.subplots(figsize=(6,4.5))
simple_bar(ax,slabels,pdrs_s,scolors,'Fig 7.7 — PDR by Season (E6)','PDR_link (%)',(94.5,97.5),pdr_link)
fig.tight_layout(); save_fig(fig,'fig_7_07_season_pdr')

  Saved: fig_7_07_season_pdr


## 8 · Fig 7.8 — CO₂ vs Rolling PDR Dual Axis (E2)  *(§7.4)*

In [33]:
dev='ED3'
g=df_link[df_link['device_id']==dev].copy().set_index('time').sort_index()
g['rx1']=1
rpdr=(g['rx1'].rolling('2h',min_periods=1).sum()/
      g['total_tx'].rolling('2h',min_periods=1).sum()*100).clip(0,100)
rco2=g['co2'].rolling('2h',min_periods=1).mean()
fig,ax1=plt.subplots(figsize=(12,4))
ax2=ax1.twinx()
ax1.plot(rpdr.index,rpdr.values,color=C['blue'],linewidth=0.5,alpha=0.8,label='PDR (%)')
ax2.plot(rco2.index,rco2.values,color=C['orange'],linewidth=0.6,alpha=0.75,label='CO₂ (ppm)')
ax1.set_ylabel('PDR (%)',color=C['blue']); ax2.set_ylabel('CO₂ (ppm)',color=C['orange'])
ax1.set_ylim(0,110); ax1.set_xlabel('Date')
ax1.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))
lines1,labs1=ax1.get_legend_handles_labels(); lines2,labs2=ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2,labs1+labs2,loc='upper right')
ax1.set_title(f'Fig 7.8 — {dev}: Rolling PDR vs CO₂ Concentration (2-hour window)')
fig.tight_layout(); save_fig(fig,'fig_7_08_co2_vs_pdr_ed3')

  Saved: fig_7_08_co2_vs_pdr_ed3


## 9 · Fig 7.9 — CO₂ Tier PDR + Burst Rate (E1)  *(§7.4)*

In [34]:
tiers=['background','moderate','high']
xlabs_co2=['Background\n(≤500 ppm)','Moderate\n(500–700 ppm)','High\n(>700 ppm)']
pdrs_co2=[pdr(df_link[df_link['e1_co2_tier']==t]) for t in tiers]
bursts_co2=[burst_rate(df_link[df_link['e1_co2_tier']==t]) for t in tiers]
x=np.arange(len(tiers)); w=0.35
fig,ax1=plt.subplots(figsize=(7,4.5)); ax2=ax1.twinx()
b1=ax1.bar(x-w/2,pdrs_co2,w,color=C['blue'],alpha=0.88,label='PDR_link (%)',edgecolor='white')
b2=ax2.bar(x+w/2,bursts_co2,w,color=C['orange'],alpha=0.88,label='Burst rate (%)',edgecolor='white')
for bar,v in zip(b1,pdrs_co2): ax1.text(bar.get_x()+bar.get_width()/2,v+0.02,f'{v:.1f}%',ha='center',va='bottom',fontsize=8)
for bar,v in zip(b2,bursts_co2): ax2.text(bar.get_x()+bar.get_width()/2,v+0.001,f'{v:.2f}%',ha='center',va='bottom',fontsize=8,color=C['orange'])
ax1.set_xticks(x); ax1.set_xticklabels(xlabs_co2)
ax1.set_ylabel('PDR_link (%)',color=C['blue']); ax2.set_ylabel('Burst rate (%)',color=C['orange'])
ax1.set_title('Fig 7.9 — CO₂ Tier: PDR and Burst Rate (E1)')
ax1.set_ylim(93,98.5); ax2.set_ylim(0,0.5)
ax1.legend(handles=[mpatches.Patch(color=C['blue'],label='PDR_link (%)'),
                     mpatches.Patch(color=C['orange'],label='Burst rate (%)')],loc='lower left')
fig.tight_layout(); save_fig(fig,'fig_7_09_co2_pdr_burst')

  Saved: fig_7_09_co2_pdr_burst


## 10 · Fig 7.10 — PM2.5 Spike + Tier PDR (E7, E8)  *(§7.5.1)*

In [35]:
fig,axes=plt.subplots(1,2,figsize=(10,4.5))
# E7: spike vs no spike
g1=df_link[df_link['e7_pm25_spike']==1]; g0=df_link[df_link['e7_pm25_spike']==0]
simple_bar(axes[0],['Spike\n(>90th pctile)','No Spike'],[pdr(g1),pdr(g0)],
           [C['red'],C['blue']],'Fig 7.10a — PM2.5 Spike PDR (E7)','PDR_link (%)',(93,97.5),pdr_link)
# E8: tier
pm_tiers=['clean','moderate','elevated']
pm_labs=['Clean\n(<2 µg/m³)','Moderate\n(2–10 µg/m³)','Elevated\n(>10 µg/m³)']
pm_pdrs=[pdr(df_link[df_link['e8_pm25_tier']==t]) if len(df_link[df_link['e8_pm25_tier']==t])>0 else 0 for t in pm_tiers]
simple_bar(axes[1],pm_labs,pm_pdrs,[C['green'],C['orange'],C['red']],
           'Fig 7.10b — PM2.5 Tier PDR (E8)','PDR_link (%)',(88,98),pdr_link)
fig.tight_layout(); save_fig(fig,'fig_7_10_pm25_pdr')

  Saved: fig_7_10_pm25_pdr


## 11 · Fig 7.11 — Pressure Tier PDR (E9)  *(§7.5.2)*

In [36]:
pt=['low','medium_low','medium_high','high']
pt_labs=['Low','Medium-Low','Medium-High','High']
pt_pdrs=[pdr(df_link[df_link['e9_pressure_tier']==t]) for t in pt]
fig,ax=plt.subplots(figsize=(6,4.5))
simple_bar(ax,pt_labs,pt_pdrs,[C['blue'],C['green'],C['orange'],C['red']],
           'Fig 7.11 — PDR by Pressure Tier (E9)','PDR_link (%)',(94.5,97.5),pdr_link)
fig.tight_layout(); save_fig(fig,'fig_7_11_pressure_tier_pdr')

  Saved: fig_7_11_pressure_tier_pdr


## 12 · Fig 7.12 — Humidity Tier PDR (E11)  *(§7.5.3)*

In [37]:
ht=['dry','normal','humid']
ht_labs=['Dry\n(<40%)','Normal\n(40–55%)','Humid\n(55–70%)']
ht_pdrs=[pdr(df_link[df_link['e11_humidity_tier']==t]) if len(df_link[df_link['e11_humidity_tier']==t])>0 else 0 for t in ht]
fig,ax=plt.subplots(figsize=(6,4.5))
simple_bar(ax,ht_labs,ht_pdrs,[C['blue'],C['green'],C['orange']],
           'Fig 7.12 — PDR by Humidity Tier (E11)','PDR_link (%)',(90,98),pdr_link)
fig.tight_layout(); save_fig(fig,'fig_7_12_humidity_tier_pdr')

  Saved: fig_7_12_humidity_tier_pdr


## 13 · Fig 7.13 — Temperature Tier PDR (E12)  *(§7.5.3)*

In [38]:
tt=['cold','cool','warm','hot']
tt_labs=['Cold','Cool','Warm','Hot']
tt_pdrs=[pdr(df_link[df_link['e12_temp_tier']==t]) for t in tt]
fig,ax=plt.subplots(figsize=(6,4.5))
simple_bar(ax,tt_labs,tt_pdrs,[C['navy'],C['blue'],C['orange'],C['red']],
           'Fig 7.13 — PDR by Temperature Tier (E12)','PDR_link (%)',(94.5,97.5),pdr_link)
fig.tight_layout(); save_fig(fig,'fig_7_13_temperature_tier_pdr')

  Saved: fig_7_13_temperature_tier_pdr


## 14 · Fig 7.14 — RSSI Tier PDR (E17)  *(§7.5)*

In [39]:
rt=['weak','moderate','strong']
rt_labs=['Weak\n(<−90 dBm)','Moderate\n(−90 to −70 dBm)','Strong\n(>−70 dBm)']
rt_pdrs=[pdr(df_link[df_link['e17_rssi_tier']==t]) for t in rt]
fig,ax=plt.subplots(figsize=(6,4.5))
simple_bar(ax,rt_labs,rt_pdrs,[C['red'],C['orange'],C['green']],
           'Fig 7.14 — PDR by RSSI Tier (E17)\n(near-identical values confirm signal quality is not primary driver)',
           'PDR_link (%)',(95,97.5),pdr_link)
fig.tight_layout(); save_fig(fig,'fig_7_14_rssi_tier_pdr')

  Saved: fig_7_14_rssi_tier_pdr


## 15 · Fig 7.15 — ESP Tier PDR (E18)  *(§7.5)*

In [40]:
et=['low_esp','medium_esp','high_esp']
et_labs=['Low ESP','Medium ESP','High ESP']
et_pdrs=[pdr(df_link[df_link['e18_esp_tier']==t]) for t in et]
fig,ax=plt.subplots(figsize=(6,4.5))
simple_bar(ax,et_labs,et_pdrs,[C['red'],C['orange'],C['green']],
           'Fig 7.15 — PDR by ESP Tier (E18)\n(no monotone pattern — signal quality does not drive losses)',
           'PDR_link (%)',(95,97.5),pdr_link)
fig.tight_layout(); save_fig(fig,'fig_7_15_esp_tier_pdr')

  Saved: fig_7_15_esp_tier_pdr


## 16 · Fig 7.16 — SF PDR + Burst Rate (E13)  *(§7.6.1)*

In [41]:
sfs=[7,8,9,10]
sf_xlabs=[f'SF{sf}\n({TOA_MS[sf]}ms)' for sf in sfs]
pdrs_sf=[pdr(df_link[df_link['e13_sf']==sf]) for sf in sfs]
bursts_sf=[burst_rate(df_link[df_link['e13_sf']==sf]) for sf in sfs]
x=np.arange(len(sfs)); w=0.35
fig,ax1=plt.subplots(figsize=(7,4.5)); ax2=ax1.twinx()
b1=ax1.bar(x-w/2,pdrs_sf,w,color=C['blue'],alpha=0.88,label='PDR_link (%)',edgecolor='white')
b2=ax2.bar(x+w/2,bursts_sf,w,color=C['orange'],alpha=0.88,label='Burst rate (%)',edgecolor='white')
for bar,v in zip(b1,pdrs_sf): ax1.text(bar.get_x()+bar.get_width()/2,v+0.02,f'{v:.1f}%',ha='center',va='bottom',fontsize=8)
for bar,v in zip(b2,bursts_sf): ax2.text(bar.get_x()+bar.get_width()/2,v+0.001,f'{v:.2f}%',ha='center',va='bottom',fontsize=8,color=C['orange'])
ax1.set_xticks(x); ax1.set_xticklabels(sf_xlabs)
ax1.set_ylabel('PDR_link (%)',color=C['blue']); ax2.set_ylabel('Burst rate (%)',color=C['orange'])
ax1.set_title('Fig 7.16 — SF PDR and Burst Rate (E13)')
ax1.set_ylim(88,100); ax2.set_ylim(0,0.8)
ax1.legend(handles=[mpatches.Patch(color=C['blue'],label='PDR_link (%)'),
                     mpatches.Patch(color=C['orange'],label='Burst rate (%)')],loc='upper right')
fig.tight_layout(); save_fig(fig,'fig_7_16_sf_pdr_burst')

  Saved: fig_7_16_sf_pdr_burst


## 17 · Fig 7.17 — SF Tier PDR Comparison (E14, E15)  *(§7.6.1)*

In [42]:
st_labs=['Low SF (7–8)\nToA ≤ 134ms','High SF (9–10)\nToA ≥ 247ms']
st_pdrs=[pdr(df_link[df_link['e14_sf_tier']==t]) for t in ['low_sf','high_sf']]
fig,ax=plt.subplots(figsize=(5,4.5))
simple_bar(ax,st_labs,st_pdrs,[C['blue'],C['red']],
           'Fig 7.17 — SF Tier PDR (E14/E15)','PDR_link (%)',(93,99),pdr_link)
fig.tight_layout(); save_fig(fig,'fig_7_17_sf_tier_pdr')

  Saved: fig_7_17_sf_tier_pdr


## 18 · Fig 7.18 — SF × CO₂ Heatmap — Headline Novel Finding  *(§7.6.2)*

In [43]:
co2_tiers=['background','moderate','high']
sf_tiers=['low_sf','high_sf']
sf_labels=['Low SF (7–8)\nToA ≤ 134ms','High SF (9–10)\nToA ≥ 247ms']
co2_labels=['Background CO₂\n(≤500 ppm)','Moderate CO₂\n(500–700 ppm)','High CO₂\n(>700 ppm)']
matrix=np.zeros((2,3))
for j,co2 in enumerate(co2_tiers):
    for i,sf in enumerate(sf_tiers):
        g=df_link[(df_link['e14_sf_tier']==sf)&(df_link['e1_co2_tier']==co2)]
        matrix[i,j]=pdr(g)
fig,ax=plt.subplots(figsize=(8,4))
im=ax.imshow(matrix,cmap='RdYlGn',aspect='auto',vmin=matrix.min()-0.5,vmax=matrix.max()+0.5)
for i in range(2):
    for j in range(3):
        v=matrix[i,j]
        ax.text(j,i,f'{v:.2f}%',ha='center',va='center',fontsize=12,fontweight='bold',
                color='white' if v<95.5 else 'black')
for j,co2 in enumerate(co2_tiers):
    gap=matrix[0,j]-matrix[1,j]
    ax.text(j,1.6,f'gap={gap:.2f} pp',ha='center',va='bottom',fontsize=8.5,color=C['red'],style='italic')
ax.set_xticks([0,1,2]); ax.set_xticklabels(co2_labels,fontsize=9)
ax.set_yticks([0,1]); ax.set_yticklabels(sf_labels,fontsize=9)
ax.set_title('Fig 7.18 — Joint PDR: SF Tier × CO₂ Tier (E14×E1)\nGap widens 2.71→4.86 pp — SF×occupancy interaction confirmed',fontsize=10)
plt.colorbar(im,ax=ax,shrink=0.75).set_label('PDR_link (%)',fontsize=9)
fig.tight_layout(); save_fig(fig,'fig_7_18_sf_co2_heatmap')

  Saved: fig_7_18_sf_co2_heatmap


## 19 · Fig 7.19 — Logistic Regression Forest Plot (Loss Risk)  *(§7.7)*

In [44]:
ev=or_table[~or_table['predictor'].str.startswith('device_id_')].copy()
ev=ev.sort_values('OR_loss',ascending=True).reset_index(drop=True)
fig,ax=plt.subplots(figsize=(9,len(ev)*0.38+1.5))
for i,row in ev.iterrows():
    color=C['red'] if row['OR_loss']>1 else C['blue']
    ax.plot([row['loss_CI_lo'],row['loss_CI_hi']],[i,i],color=color,lw=1.5,alpha=0.7,solid_capstyle='round')
    ax.scatter(row['OR_loss'],i,color=color,s=40,zorder=5)
ax.axvline(x=1.0,color='black',lw=0.8,ls='--',alpha=0.6)
ax.set_yticks(range(len(ev))); ax.set_yticklabels(ev['predictor'],fontsize=7.5)
ax.set_xlabel('Odds Ratio — loss risk (log scale)')
ax.set_title('Fig 7.19 — Loss Risk OR with 95% CI (event predictors only)')
ax.set_xscale('log'); ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
ax.legend(handles=[mpatches.Patch(color=C['red'],label='↑ higher risk (OR>1)'),
                    mpatches.Patch(color=C['blue'],label='↓ lower risk (OR<1)')],loc='lower right',fontsize=8)
fig.tight_layout(); save_fig(fig,'fig_7_19_forest_plot_loss')

  Saved: fig_7_19_forest_plot_loss


## 20 · Fig 7.20 — Loss vs Burst OR Comparison  *(§7.7)*

In [45]:
ev2=or_table[~or_table['predictor'].str.startswith('device_id_')].copy()
ev2=ev2.sort_values('OR_loss',ascending=False).head(12).reset_index(drop=True)
x=np.arange(len(ev2)); w=0.35
fig,ax=plt.subplots(figsize=(10,5))
b1=ax.bar(x-w/2,ev2['OR_loss'],w,color=C['blue'],alpha=0.88,label='OR (loss risk)',edgecolor='white')
b2=ax.bar(x+w/2,ev2['OR_burst'],w,color=C['red'],alpha=0.88,label='OR (burst risk)',edgecolor='white')
ax.axhline(y=1.0,color='black',lw=0.8,ls='--',alpha=0.6)
ax.set_xticks(x); ax.set_xticklabels(ev2['predictor'],rotation=45,ha='right',fontsize=7.5)
ax.set_ylabel('Odds Ratio')
ax.set_title('Fig 7.20 — Loss Risk vs Burst Risk OR (top 12 predictors)')
ax.legend()
fig.tight_layout(); save_fig(fig,'fig_7_20_loss_vs_burst_or')

  Saved: fig_7_20_loss_vs_burst_or


## 21 · Summary

In [46]:
print('All figures generated:')
print()
for f in sorted(FIG_DIR.glob('fig_*.pdf')):
    print(f'  {f.name}')
print(f'\nTotal: {len(list(FIG_DIR.glob("fig_*.pdf")))} figures')
print(f'Folder: {FIG_DIR.resolve()}')

All figures generated:

  fig_7_01_pdr_decomposition.pdf
  fig_7_02_loss_zone_breakdown.pdf
  fig_7_03_rolling_pdr.pdf
  fig_7_04_burst_distribution.pdf
  fig_7_05_weekday_weekend_pdr.pdf
  fig_7_06_time_of_day_pdr.pdf
  fig_7_07_season_pdr.pdf
  fig_7_08_co2_vs_pdr_ed3.pdf
  fig_7_09_co2_pdr_burst.pdf
  fig_7_10_pm25_pdr.pdf
  fig_7_11_pressure_tier_pdr.pdf
  fig_7_12_humidity_tier_pdr.pdf
  fig_7_13_temperature_tier_pdr.pdf
  fig_7_14_rssi_tier_pdr.pdf
  fig_7_15_esp_tier_pdr.pdf
  fig_7_16_sf_pdr_burst.pdf
  fig_7_17_sf_tier_pdr.pdf
  fig_7_18_sf_co2_heatmap.pdf
  fig_7_19_forest_plot_loss.pdf
  fig_7_20_loss_vs_burst_or.pdf

Total: 20 figures
Folder: D:\Thesis\lorawan_master_thesis\figures
